# Synthèse des résultats

---
## 1. Imports et chargement des données

In [1]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score, confusion_matrix,
)

from html import escape
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=UserWarning)

MODE_AFFICHAGE = "html"

DATA_DIR   = Path("../data")
MODELS_DIR = Path("../models")

X_train = pd.read_csv(DATA_DIR / "weatherAUS_X_train.csv")
X_test  = pd.read_csv(DATA_DIR / "weatherAUS_X_test.csv")

y_train = pd.read_csv(DATA_DIR / "weatherAUS_y_train.csv").squeeze("columns").astype(int)
y_test  = pd.read_csv(DATA_DIR / "weatherAUS_y_test.csv").squeeze("columns").astype(int)

# Scaler de secours : certains modèles ont été entraînés sur des données standardisées
# sans que le StandardScaler correspondant soit sérialisé (cf. linear_svc_baseline).
# Le réajuster sur le train reproduit exactement la transformation d'origine.
SCALER_SECOURS = StandardScaler().fit(X_train)

RESULTATS = {}

print(f"X_train : {X_train.shape[0]:,} lignes x {X_train.shape[1]} colonnes")
print(f"X_test  : {X_test.shape[0]:,} lignes x {X_test.shape[1]} colonnes")
print(f"Taux de pluie train : {y_train.mean():.2%} | test : {y_test.mean():.2%}")

X_train : 113,506 lignes x 23 colonnes
X_test  : 25,314 lignes x 23 colonnes
Taux de pluie train : 21.96% | test : 22.69%


---
## 2. Fonctions utilitaires

Les bundles n'ont pas tous la même forme selon le notebook qui les a produits :

| Origine | Contenu du fichier | Clé du seuil |
|---|---|---|
| `light_gbm`, `linear_svc` — baseline / grid search | modèle nu (pas de dictionnaire) | — |
| `light_gbm`, `linear_svc` — threshold | `{model, seuil, features}` | `seuil` |
| `logistic_regression` | `{model, threshold, features, scaler, metrics, ...}` | `threshold` |

Les fonctions ci-dessous normalisent ces variantes et gèrent les cas particuliers :
`Pipeline` (scaling interne), scaler externe embarqué, termes d'interaction à
reconstruire (`poly` / `top_k` / `cross_idx`), et ensemble bootstrap (`model` est une
**liste** de modèles dont on moyenne les probabilités).

In [2]:
def normaliser_bundle(objet):
    """Ramène un fichier sérialisé (modèle nu ou dictionnaire) à un dictionnaire homogène."""
    if isinstance(objet, dict) and "model" in objet:
        bundle = dict(objet)
    else:
        bundle = {"model": objet}
    # Uniformisation de la clé du seuil : 'seuil' (FR) et 'threshold' (EN) coexistent
    if "threshold" not in bundle and "seuil" in bundle:
        bundle["threshold"] = bundle["seuil"]
    return bundle


def _modele_reference(modele):
    """Renvoie un modèle représentatif (1er élément si c'est un ensemble bootstrap)."""
    return modele[0] if isinstance(modele, list) else modele


def preparer_X(bundle, X):
    """Applique au jeu X exactement le prétraitement attendu par le modèle rechargé."""
    modele = bundle["model"]
    ref    = _modele_reference(modele)
    notes  = []

    if isinstance(ref, Pipeline):
        Xp = X.copy()
        notes.append("Pipeline : scaling interne")
    else:
        scaler = bundle.get("scaler")
        if scaler is not None:
            notes.append("scaler du bundle")
        elif hasattr(ref, "coef_"):
            # Modèle linéaire sans scaler sérialisé → il a forcément été entraîné standardisé
            scaler = SCALER_SECOURS
            notes.append("scaler ABSENT du bundle → réajusté sur le train")
        if scaler is not None:
            Xp = pd.DataFrame(scaler.transform(X), columns=X.columns, index=X.index)
        else:
            Xp = X.copy()
            notes.append("features brutes")

    # Reconstruction des termes d'interaction si le bundle les décrit
    if "poly" in bundle and "top_k" in bundle:
        arr = bundle["poly"].transform(Xp[bundle["top_k"]])
        croix = pd.DataFrame(arr[:, bundle["cross_idx"]],
                             columns=bundle["cross_names"], index=Xp.index)
        Xp = pd.concat([Xp, croix], axis=1)
        notes.append("termes d'interaction reconstruits")

    # Réalignement sur la liste ordonnée de variables du bundle
    features = bundle.get("features")
    if features is not None and list(Xp.columns) != list(features):
        manquantes = [f for f in features if f not in Xp.columns]
        if manquantes:
            raise ValueError(f"{len(manquantes)} variable(s) attendue(s) introuvable(s) : {manquantes[:5]}")
        Xp = Xp[list(features)]
        notes.append("colonnes réalignées")

    return Xp, notes


def calculer_scores(modele, Xp):
    """Renvoie (scores de la classe positive, type de score)."""
    if isinstance(modele, list):                       # ensemble bootstrap
        return np.mean([m.predict_proba(Xp)[:, 1] for m in modele], axis=0), "proba"
    if hasattr(modele, "predict_proba"):
        return modele.predict_proba(Xp)[:, 1], "proba"
    if hasattr(modele, "decision_function"):
        return modele.decision_function(Xp), "decision"
    raise TypeError(f"{type(modele).__name__} n'expose ni predict_proba ni decision_function")


def resoudre_seuil(bundle, type_score):
    """Renvoie (seuil, origine, alerte éventuelle)."""
    defaut = 0.5 if type_score == "proba" else 0.0
    seuil  = bundle.get("threshold")
    if seuil is None:
        return defaut, "défaut", None
    alerte = None
    if type_score == "proba" and not (0.0 < seuil < 1.0):
        alerte = (f"seuil {seuil} hors de [0, 1] alors que le score est une PROBABILITÉ "
                  f"→ toutes les prédictions basculent dans une seule classe")
    return seuil, "bundle", alerte

### 2.1. Affichage et évaluation

Le format du tableau reprend celui utilisé dans les notebooks de modélisation.

In [3]:
SEP = "─" * 52

_STYLE = ("font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12.5px; "
          "line-height: 1.35; white-space: pre; margin: 0 0 14px 0;")


def emettre(bloc):
    """Émet un bloc de texte comme sortie autonome (HTML) ou via stdout."""
    if MODE_AFFICHAGE == "html":
        display(HTML(f"<pre style='{_STYLE}'>{escape(bloc)}</pre>"))
    else:
        print(bloc + "\n")


def afficher_evaluation(nom_classe, variante, m, seuil, origine_seuil, type_score, notes, alerte):
    """Construit et émet le tableau de métriques Train / Test."""
    label_score = "probabilité" if type_score == "proba" else "decision_function"
    lignes = [
        SEP,
        f"{'ÉVALUATION DU MODÈLE — ' + nom_classe:^52}",
        f"{variante:^52}",
        SEP,
        f"{'Métrique':<28} {'Train':>10} {'Test':>10}",
        SEP,
        f"{'F1-score (classe Yes)':<28} {m['f1_train']:>10.4f} {m['f1_test']:>10.4f}",
        SEP,
        f"{'Recall   (classe Yes)':<28} {m['recall_train']:>10.4f} {m['recall_test']:>10.4f}",
        f"{'Précision (classe Yes)':<28} {m['prec_train']:>10.4f} {m['prec_test']:>10.4f}",
        SEP,
        f"{'AUC-ROC':<28} {m['auc_train']:>10.4f} {m['auc_test']:>10.4f}",
        f"{'PR-AUC':<28} {m['pr_train']:>10.4f} {m['pr_test']:>10.4f}",
        SEP,
        f"{'Seuil de décision':<28} {seuil:>10} ({origine_seuil}, {label_score})",
        f"{'Prétraitement':<28} {' + '.join(notes) if notes else 'aucun':>10}",
    ]
    if alerte:
        lignes += ["", f"  /!\\  ALERTE — {alerte}"]
    lignes.append(SEP)
    emettre("\n".join(lignes))


def classer_surapprentissage(ecart):
    """Classification de l'écart Train/Test sur l'AUC-ROC (convention du rapport)."""
    if ecart < 0.02:
        return "non"
    if ecart < 0.05:
        return "léger"
    if ecart < 0.10:
        return "modéré"
    return "fort"


def evaluer_fichier(chemin, dossier):
    """Recharge un fichier, réévalue le modèle, affiche le tableau et renvoie une ligne de synthèse."""
    bundle = normaliser_bundle(joblib.load(chemin))
    modele = bundle["model"]
    ref    = _modele_reference(modele)

    nom_classe = type(ref).__name__
    if isinstance(ref, Pipeline):
        nom_classe = f"Pipeline({type(ref.steps[-1][1]).__name__})"
    if isinstance(modele, list):
        nom_classe = f"Ensemble[{len(modele)} x {nom_classe}]"

    Xtr, notes = preparer_X(bundle, X_train)
    Xte, _     = preparer_X(bundle, X_test)

    s_train, type_score = calculer_scores(modele, Xtr)
    s_test,  _          = calculer_scores(modele, Xte)

    seuil, origine_seuil, alerte = resoudre_seuil(bundle, type_score)

    p_train = (s_train >= seuil).astype(int)
    p_test  = (s_test  >= seuil).astype(int)

    m = {
        "f1_train":     f1_score(y_train, p_train, pos_label=1, zero_division=0),
        "f1_test":      f1_score(y_test,  p_test,  pos_label=1, zero_division=0),
        "recall_train": recall_score(y_train, p_train, zero_division=0),
        "recall_test":  recall_score(y_test,  p_test,  zero_division=0),
        "prec_train":   precision_score(y_train, p_train, zero_division=0),
        "prec_test":    precision_score(y_test,  p_test,  zero_division=0),
        "auc_train":    roc_auc_score(y_train, s_train),
        "auc_test":     roc_auc_score(y_test,  s_test),
        "pr_train":     average_precision_score(y_train, s_train),
        "pr_test":      average_precision_score(y_test,  s_test),
    }

    variante = chemin.stem
    afficher_evaluation(nom_classe, variante, m, seuil, origine_seuil, type_score, notes, alerte)

    ecart = m["auc_train"] - m["auc_test"]
    return (dossier, variante), {
        "Famille": dossier, "Variante": variante, "Modèle": nom_classe,
        "Seuil": seuil, "Type score": type_score,
        "F1 Train": m["f1_train"], "F1 Test": m["f1_test"],
        "Recall Test": m["recall_test"], "Précision Test": m["prec_test"],
        "AUC-ROC Train": m["auc_train"], "AUC-ROC Test": m["auc_test"],
        "PR-AUC Test": m["pr_test"],
        "Écart AUC": ecart, "Surapprentissage": classer_surapprentissage(ecart),
        "Alerte": alerte or "",
    }


def evaluer_dossier(dossier):
    """Évalue tous les modèles d'un sous-dossier de ../models/ et met à jour RESULTATS."""
    chemin_dossier = MODELS_DIR / dossier
    if not chemin_dossier.is_dir():
        print(f"Dossier introuvable : {chemin_dossier}")
        return

    fichiers = sorted(list(chemin_dossier.glob("*.pkl")) + list(chemin_dossier.glob("*.joblib")))
    if not fichiers:
        print(f"Aucun modèle sérialisé dans {chemin_dossier}")
        return

    emettre("\n".join(["#" * 52,
                       f"#  {dossier.upper()}  —  {len(fichiers)} modèle(s)",
                       "#" * 52]))

    for fichier in fichiers:
        try:
            cle, ligne = evaluer_fichier(fichier, dossier)
            RESULTATS[cle] = ligne
        except Exception as e:
            emettre("\n".join([SEP,
                               f"  ÉCHEC — {fichier.name}",
                               f"  {type(e).__name__} : {e}",
                               SEP]))

---
## 3. Inventaire des modèles disponibles

In [4]:
if not MODELS_DIR.is_dir():
    print(f"Dossier introuvable : {MODELS_DIR.resolve()}")
else:
    sous_dossiers = sorted(d for d in MODELS_DIR.iterdir() if d.is_dir())
    print(f"{len(sous_dossiers)} famille(s) de modèles dans {MODELS_DIR.resolve()}\n")

    for d in sous_dossiers:
        fichiers = sorted(list(d.glob("*.pkl")) + list(d.glob("*.joblib")))
        lignes = [f"{d.name}/  ({len(fichiers)} fichier(s))"]
        for f in fichiers:
            taille = f.stat().st_size / 1024
            try:
                objet = joblib.load(f)
                if isinstance(objet, dict):
                    structure = "dict → " + ", ".join(sorted(objet.keys()))
                    contenu   = type(objet.get("model")).__name__
                else:
                    structure = "modèle nu"
                    contenu   = type(objet).__name__
                lignes.append(f"   {f.name:<44} {taille:>8.1f} Ko   [{contenu}]")
                lignes.append(f"   {'':<44} {'':>8}      {structure}")
            except Exception as e:
                lignes.append(f"   {f.name:<44} {taille:>8.1f} Ko   ILLISIBLE : {type(e).__name__}")
        emettre("\n".join(lignes))

3 famille(s) de modèles dans /home/nicolas/Bureau/alternance/juin25_alt2_mle_meteo/models



---
## 4. Évaluation par famille de modèles

### 4.1. Régression logistique — `logistic_regression`

In [5]:
evaluer_dossier("logistic_regression")

### 4.2. SVM linéaire — `linear_svc`

In [6]:
evaluer_dossier("linear_svc")

### 4.3. Gradient boosting — `light_gbm`

In [7]:
evaluer_dossier("light_gbm")

### 4.4. Arbre de décision — `decision_tree`

In [8]:
evaluer_dossier("decision_tree")

Dossier introuvable : ../models/decision_tree


### 4.5. Forêt aléatoire — `random_forest`

In [9]:
evaluer_dossier("random_forest")

Dossier introuvable : ../models/random_forest


### 4.6. XGBoost — `xgboost`

In [10]:
evaluer_dossier("xgboost")

Dossier introuvable : ../models/xgboost


### 4.7. CatBoost — `catboost`

In [11]:
evaluer_dossier("catboost")

Dossier introuvable : ../models/catboost


---
## 5. Tableau récapitulatif

Synthèse de toutes les variantes évaluées, triées par AUC-ROC de test décroissante.
L'AUC-ROC gouverne la comparaison entre modèles ; le F1 sur la classe *Yes* mesure la performance au point de fonctionnement retenu.

In [38]:
if not RESULTATS:
    print("Aucun résultat — exécutez d'abord les cellules de la section 4.")
else:
    recap = pd.DataFrame(RESULTATS.values())

    # Compromis F1 / Recall : même logique de départage que dans le tableau
    # "meilleure variante par famille" (cellule 29)
    recap["Compromis F1/Recall"] = (recap["F1 Test"] + recap["Recall Test"]) / 2

    # Grouper par famille : chaque famille est ordonnée par sa meilleure
    # AUC-ROC Test, puis les variantes sont triées à l'intérieur par
    # AUC-ROC Test, Recall Test, F1 Test décroissants
    ordre_familles = (recap.groupby("Famille")["AUC-ROC Test"].max()
                            .sort_values(ascending=False).index)
    recap["Famille"] = pd.Categorical(recap["Famille"], categories=ordre_familles, ordered=True)
    recap = (recap.sort_values(["Famille", "AUC-ROC Test", "Recall Test", "F1 Test"],
                                ascending=[True, False, False, False])
                  .reset_index(drop=True))

    colonnes = ["Famille", "Variante", "Seuil", "F1 Train", "F1 Test",
                "Recall Test", "Précision Test", "Compromis F1/Recall",
                "AUC-ROC Train", "AUC-ROC Test", "PR-AUC Test", "Écart AUC",
                "Surapprentissage"]

    display(recap[["Famille", "Variante", "Seuil", "F1 Train", "F1 Test",
                   "Recall Test", "Précision Test", "AUC-ROC Train", 
                   "AUC-ROC Test", "PR-AUC Test", "Écart AUC", "Surapprentissage"]]
            .style.format({c: "{:.4f}" for c in colonnes if c not in
                     ["Famille", "Variante", "Seuil", "Surapprentissage"]})
            .background_gradient(subset=["AUC-ROC Test", "Recall Test", "F1 Test"], cmap="Greens"))

    alertes = recap[recap["Alerte"] != ""]
    if not alertes.empty:
        print(f"\n{len(alertes)} modèle(s) signalé(s) :")
        for _, r in alertes.iterrows():
            print(f"  - {r['Famille']}/{r['Variante']} : {r['Alerte']}")

,Famille,Variante,Seuil,F1 Train,F1 Test,Recall Test,Précision Test,AUC-ROC Train,AUC-ROC Test,PR-AUC Test,Écart AUC,Surapprentissage
0,light_gbm,light_gbm_threshold,0.400000,0.6766,0.6156,0.8519,0.4819,0.9324,0.8800,0.7232,0.0524,modéré
1,light_gbm,light_gbm_randomized_search,0.500000,0.7079,0.6436,0.7901,0.5430,0.9324,0.8800,0.7232,0.0524,modéré
2,light_gbm,light_gbm_baseline,0.500000,0.7548,0.6429,0.7673,0.5532,0.9550,0.8776,0.7207,0.0774,modéré
3,logistic_regression,logistic_regression_interactions,0.670000,0.6436,0.6393,0.6461,0.6326,0.8801,0.8675,0.6965,0.0126,non
4,logistic_regression,logistic_regression_imbalance_balanced,0.500000,0.6277,0.6201,0.7775,0.5156,0.8761,0.8634,0.6859,0.0127,non
5,logistic_regression,logistic_regression_bootstrap_bagged,0.670000,0.6376,0.6340,0.6367,0.6312,0.8761,0.8633,0.6860,0.0127,non
6,logistic_regression,logistic_regression_grid_search_tuned,0.670000,0.6376,0.6342,0.6371,0.6313,0.8761,0.8633,0.6859,0.0128,non
7,logistic_regression,logistic_regression_imbalance_default,0.500000,0.6012,0.6004,0.5159,0.7180,0.8756,0.8629,0.6883,0.0128,non
8,logistic_regression,logistic_regression_baseline,0.500000,0.6013,0.6004,0.5159,0.7178,0.8756,0.8629,0.6883,0.0128,non
9,logistic_regression,logistic_regression_imbalance_smote,0.500000,0.6263,0.6195,0.7777,0.5147,0.8758,0.8628,0.6870,0.0129,non


Le tableau est ordonné par AUC-ROC Test décroissante, puis par un compromis F1/Recall (moyenne simple)

### 5.1. Meilleure variante par famille

In [39]:
if RESULTATS:
    recap = (pd.DataFrame(RESULTATS.values())
             .sort_values("AUC-ROC Test", ascending=False)
             .reset_index(drop=True))

    # Critère de départage entre variantes à AUC-ROC Test égale :
    # compromis F1 / Recall (moyenne simple), pour ne pas ignorer le recall
    recap["Compromis F1/Recall"] = (recap["F1 Test"] + recap["Recall Test"]) / 2

    meilleurs = (recap.sort_values(["AUC-ROC Test", "Compromis F1/Recall"],
                                    ascending=[False, False])
                      .groupby("Famille", as_index=False).first()
                      .sort_values("AUC-ROC Test", ascending=False))

    display(meilleurs[["Famille", "Variante", "Modèle", "F1 Test", "Recall Test",
                       "Précision Test", "AUC-ROC Test", "PR-AUC Test",
                       "Surapprentissage"]]
            .style.format({c: "{:.4f}" for c in
                           ["F1 Test", "Recall Test",
                            "Précision Test", "AUC-ROC Test", "PR-AUC Test"]})
            .background_gradient(subset=["AUC-ROC Test", "Recall Test", "F1 Test"], cmap="Greens"))

,Famille,Variante,Modèle,F1 Test,Recall Test,Précision Test,AUC-ROC Test,PR-AUC Test,Surapprentissage
0,light_gbm,light_gbm_threshold,LGBMClassifier,0.6156,0.8519,0.4819,0.8800,0.7232,modéré
2,logistic_regression,logistic_regression_interactions,LogisticRegression,0.6393,0.6461,0.6326,0.8675,0.6965,non
1,linear_svc,linear_svc_threshold,Pipeline(LinearSVC),0.6130,0.7983,0.4975,0.8637,0.6843,non


### 5.2. Export

Le tableau est exporté en CSV pour être repris directement dans le rapport.

In [17]:
if RESULTATS:
    EXPORT_PATH = Path("../reports/synthese_modeles.csv")
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    recap.to_csv(EXPORT_PATH, index=False)
    print(f"Exporté → {EXPORT_PATH}  ({len(recap)} lignes)")

Exporté → ../reports/synthese_modeles.csv  (13 lignes)
